In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
from pyspark.sql.window import Window

##Load silver orders

In [0]:
df_silver = spark.table(SILVER_ORDERS).select("device_type").distinct()
print(f"Silver rows: {df_silver.count()}")
display(df_silver)

##Build dim_customer

In [0]:
df_dim_device_type = (df_silver
    .filter(F.col("device_type").isNotNull())
    .filter(F.col("device_type") != "")

    # Surrogate key 
    .withColumn("device_type_id",
        F.concat(
            F.lit("DATA1"),
            F.lpad(
                F.row_number().over(
                    Window.partitionBy(F.lit(1)).orderBy("device_type")
                ).cast("string"),
                5, "0"
            )
        )
    )

    .withColumn("_gold_ingested_at",
        F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat()))
    )

    .select(
        "device_type_id",
        "device_type",
        "_gold_ingested_at"
    )
)

print(f"dim_device_type rows: {df_dim_device_type.count()}")
display(df_dim_device_type)

##Write to gold

In [0]:
(df_dim_device_type
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_DEVICE_TYPE)
)

print(f"✅ {df_dim_device_type.count()} rows written to {GOLD_DEVICE_TYPE}")

##Validate

In [0]:
df_check = spark.table(GOLD_DEVICE_TYPE)

print(f"Total rows   : {df_check.count()}")
print(f"Distinct IDs : {df_check.select('device_type_id').distinct().count()}")
df_check.show(5, truncate=False)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.gold.dim_device_type;